# Workbook 2 — Beam TestStream (optional dependency)

Run from repository root after `pip install -r requirements-beam.txt`. Requires a compatible Beam SDK. **No GCP account required.** This exercise runs DirectRunner, not DataflowRunner.


In [ ]:
from pathlib import Path
import sys,json
root=Path.cwd()
if not (root/"tests/fixtures/trades.jsonl").exists():
    raise RuntimeError("Start notebook from repository root")
sys.path.insert(0,str(root/"src"))
from market_pulse.beam_pipeline import build_windowed
import apache_beam as beam
from apache_beam.testing.test_pipeline import TestPipeline
from apache_beam.testing.test_stream import TestStream
from apache_beam.testing.util import assert_that, equal_to
from apache_beam.transforms.window import TimestampedValue
events=[json.loads(s) for s in (root/"tests/fixtures/trades.jsonl").read_text().splitlines() if s.strip()]


## Build a simulated unbounded stream
`event_time_ms` explicitly stamps each event. `advance_watermark_to_infinity()` closes windows in the test.

In [ ]:
start=events[0]["event_time_ms"]//1000
stream=(TestStream().advance_watermark_to(start-2)
        .add_elements([TimestampedValue(e,e["event_time_ms"]/1000) for e in events])
        .advance_watermark_to_infinity())
expected=[("BTCUSDT",4,1793354400000),("ETHUSDT",1,1793354400000),
          ("BTCUSDT",1,1793354460000),("ETHUSDT",1,1793354460000)]
with TestPipeline(options=beam.options.pipeline_options.PipelineOptions(streaming=True)) as p:
    rows=p | "Trades" >> stream | "OHLCV" >> build_windowed()
    summaries=rows | "Summary" >> beam.Map(lambda x:(x["symbol"],x["trade_count"],x["window_start_ms"]))
    assert_that(summaries,equal_to(expected))
print("Beam TestStream: PASS")


## Questions
1. Where do event-time and processing-time differ?
2. What does allowed lateness mean if watermark already passed a window?
3. Does the fixed test prove sustained 24×7 production streaming? **No.**
4. Why might state size and shuffle volume grow for high-cardinality symbols / trade IDs?
